## Imports

In [1]:
library(dplyr)
library(Seurat)
library(patchwork)
library(ggplot2)
library(Matrix)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




In [2]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /coh_labs/mvandenbrink/users/pkaur/miniconda3/envs/seurat/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: America/Phoenix
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] Matrix_1.6-5       ggplot2_3.5.2      patchwork_1.3.2    Seurat_5.3.0      
[5] SeuratObject_5.2.0 sp_2.2-0           dplyr_1.1.4       

loaded via a namespace (and not attached):


In [3]:
base_dir <- "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files"   # <-- CHANGE THIS

pool_names <- c('bm_sample_filtered_feature_bc_matrix.h5',
                'liver_sample_filtered_feature_bc_matrix.h5',
                'lung_sample_filtered_feature_bc_matrix.h5',
                'skin_sample_filtered_feature_bc_matrix.h5',
               'spleen_sample_filtered_feature_bc_matrix.h5',
                'thymus_sample_filtered_feature_bc_matrix.h5')

tissue_labels <- c("bm",
                    "liver",
                    "lung",
                    "skin",
                    "spleen",
                    "thymus")

In [4]:
h5_paths <- file.path(base_dir, pool_names)
h5_paths

[1] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/bm_sample_filtered_feature_bc_matrix.h5"    
[2] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/liver_sample_filtered_feature_bc_matrix.h5" 
[3] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/lung_sample_filtered_feature_bc_matrix.h5"  
[4] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/skin_sample_filtered_feature_bc_matrix.h5"  
[5] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/spleen_sample_filtered_feature_bc_matrix.h5"
[6] "/coh_labs/mvandenbrink/users/jonpierce/tff1/scRNA/h5Files/h5Files/thymus_sample_filtered_feature_bc_matrix.h5"

In [5]:
expected_hto_ids <- c('TotalSeq-C0301_anti-mouse_Hashtag_1_Antibody',
                      'TotalSeq-C0302_anti-mouse_Hashtag_2_Antibody',
                      'TotalSeq-C0303_anti-mouse_Hashtag_3_Antibody')

In [6]:
hto_override <- list(
  "skin" = c(
    "TotalSeq-C0302_anti-mouse_Hashtag_2_Antibody",
    "TotalSeq-C0303_anti-mouse_Hashtag_3_Antibody"
  )
)

## Process Hashtags

In [7]:
process_pool <- function(h5_path, pool_name, tissue_label, hto_ids) {

  mat <- Read10X_h5(h5_path)

  if (!is.list(mat)) {
    stop(paste0("Only one feature type found in ", h5_path,
                ". Expected multi feature types (GEX + Antibody Capture)."))
  }
  if (!all(c("Gene Expression", "Antibody Capture") %in% names(mat))) {
    stop(paste0("Missing expected feature types in ", h5_path,
                ". Found: ", paste(names(mat), collapse = ", ")))
  }

  gex_counts <- mat[["Gene Expression"]]
  hto_counts <- mat[["Antibody Capture"]]

  # Try exact match; if not present, try partial match (from your notebook)
  if (!all(hto_ids %in% rownames(hto_counts))) {
    matched <- sapply(hto_ids, function(id) grep(id, rownames(hto_counts), value = TRUE)[1])
    if (any(is.na(matched))) {
      stop(paste0(
        "Could not match HTO IDs.\nFound HTO rows:\n  ",
        paste(rownames(hto_counts), collapse = ", "),
        "\nExpected:\n  ", paste(hto_ids, collapse = ", ")
      ))
    }
    hto_ids_use <- matched
  } else {
    hto_ids_use <- hto_ids
  }

  # Shared barcodes between GEX and HTO
  shared_cells <- intersect(colnames(gex_counts), colnames(hto_counts))
  gex_counts <- gex_counts[, shared_cells, drop = FALSE]
  hto_counts <- hto_counts[hto_ids_use, shared_cells, drop = FALSE]

  # Create Seurat object from RAW RNA counts (NO RNA normalization)
  sobj <- CreateSeuratObject(
    counts = gex_counts,
    project = pool_name,
    min.cells = 3,
    min.features = 200
  )

  sobj$tissue <- tissue_label
  sobj$pool   <- pool_name
  sobj$pct_mito <- PercentageFeatureSet(sobj, pattern = "^mt-")  # mouse

  # Add HTO assay
  sobj[["HTO"]] <- CreateAssayObject(counts = hto_counts[, colnames(sobj), drop = FALSE])

  # Remove cells with zero total HTO counts (HTODemux can fail otherwise)
  hto_colsums <- colSums(GetAssayData(sobj, assay = "HTO", layer = "counts"))
  if (any(hto_colsums == 0)) {
    sobj <- subset(sobj, cells = names(hto_colsums[hto_colsums > 0]))
  }

  # Normalize ONLY HTO assay (required for demux)
  sobj <- NormalizeData(sobj, assay = "HTO", normalization.method = "CLR", margin = 2)

  # Demux
  sobj <- HTODemux(sobj, assay = "HTO", positive.quantile = 0.99)

  return(sobj)
}

In [8]:
seurat_list <- list()

for (i in seq_along(pool_names)) {

  tissue <- tissue_labels[i]

  hto_ids_this_pool <- if (tissue %in% names(hto_override)) {
    cat("NOTE:", tissue, "using HTO override\n")
    hto_override[[tissue]]
  } else {
    expected_hto_ids
  }

  cat("\n--- Processing", tissue, "---\n")

  sobj <- tryCatch({
    process_pool(
      h5_path      = h5_paths[i],
      pool_name    = pool_names[i],
      tissue_label = tissue,
      hto_ids      = hto_ids_this_pool
    )
  }, error = function(e) {
    cat("ERROR:", tissue, "->", conditionMessage(e), "\n")
    return(NULL)
  })

  if (!is.null(sobj)) seurat_list[[tissue]] <- sobj
}

cat("\nProcessed tissues:", paste(names(seurat_list), collapse = ", "), "\n")


--- Processing bm ---


Genome matrix has multiple modalities, returning a list of matrices for this genome

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“The `slot` argument of `SetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Normalizing across cells

As of Seurat v5, we recommend using AggregateExpression to perform pseudo-bulk analysis.
This message is displayed once per session.
First grou


--- Processing liver ---


Genome matrix has multiple modalities, returning a list of matrices for this genome

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Normalizing across cells

Cutoff for TotalSeq-C0301-anti-mouse-Hashtag-1-Antibody : 29 reads

Cutoff for TotalSeq-C0302-anti-mouse-Hashtag-2-Antibody : 0 reads

Cutoff for TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody : 7 reads




--- Processing lung ---


...


--- Processing spleen ---


Genome matrix has multiple modalities, returning a list of matrices for this genome

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Normalizing across cells

Cutoff for TotalSeq-C0301-anti-mouse-Hashtag-1-Antibody : 81 reads

Cutoff for TotalSeq-C0302-anti-mouse-Hashtag-2-Antibody : 129 reads

Cutoff for TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody : 112 reads




--- Processing thymus ---


Genome matrix has multiple modalities, returning a list of matrices for this genome

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Normalizing across cells

Cutoff for TotalSeq-C0301-anti-mouse-Hashtag-1-Antibody : 96 reads

Cutoff for TotalSeq-C0302-anti-mouse-Hashtag-2-Antibody : 61 reads

Cutoff for TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody : 115 reads




Processed tissues: bm, liver, lung, skin, spleen, thymus 


## Demux Plots

In [9]:
qc_dir <- "demux_QC_plots"
dir.create(qc_dir, showWarnings = FALSE)

for (tissue in names(seurat_list)) {

  sobj <- seurat_list[[tissue]]

  # HTO feature names (2 for skin, 3 for others)
  hto_feats <- rownames(sobj[["HTO"]])

  pdf(file.path(qc_dir, paste0(tissue, "_demux_QC.pdf")), width = 11, height = 8)

  # -------------------------
  # 1) Ridge plot (HTO CLR values)
  # -------------------------
  DefaultAssay(sobj) <- "HTO"
  Idents(sobj) <- "HTO_maxID"

  print(
    RidgePlot(sobj, features = hto_feats, ncol = 1) +
      ggtitle(paste0(tissue, " – HTO Ridge Plot"))
  )

  # -------------------------
  # 2) Heatmap (USE slot='data' to avoid scale.data error)
  # -------------------------
  # This is the direct fix for your error.
  print(
    DoHeatmap(
      sobj,
      features = hto_feats,
      group.by = "HTO_maxID",
      slot = "data"         # <-- IMPORTANT (CLR normalized HTO lives here)
    ) +
      ggtitle(paste0(tissue, " – HTO Heatmap"))
  )

  # -------------------------
  # 3) ONE violin plot (matches your reference: nCount_RNA by demux class)
  # -------------------------
  DefaultAssay(sobj) <- "RNA"

  print(
    VlnPlot(
      sobj,
      features = "nCount_RNA",
      group.by = "HTO_classification.global",
      pt.size = 0.1,
      log = TRUE
    ) +
      ggtitle(paste0(tissue, " – nCount_RNA by Demux Class")) +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
  )

  # -------------------------
  # 4) Three HTO scatter plots (pairwise)
  # -------------------------
  DefaultAssay(sobj) <- "HTO"

  # Helper to reduce repetition
  plot_scatter <- function(f1, f2) {
    FeatureScatter(
      sobj,
      feature1 = f1,
      feature2 = f2,
      group.by = "HTO_classification.global"
    ) + ggtitle(paste0(tissue, " – HTO Scatter"))
  }

  if (length(hto_feats) >= 2) {
    print(plot_scatter(hto_feats[1], hto_feats[2]))
  }
  if (length(hto_feats) == 3) {
    print(plot_scatter(hto_feats[1], hto_feats[3]))
    print(plot_scatter(hto_feats[2], hto_feats[3]))
  }

  dev.off()

  # save back (optional)
  seurat_list[[tissue]] <- sobj
}


Warning message:
“The `slot` argument of `FetchData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Warning message:
“`PackageCheck()` was deprecated in SeuratObject 5.0.0.
ℹ Please use `rlang::check_installed()` instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Picking joint bandwidth of 0.0293

Picking joint bandwidth of 0.0344

Picking joint bandwidth of 0.0296

Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'bm – HTO Ridge Plot' in 'mbcsToSbcs': dot substituted for <e2>”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'bm – HTO Ridge Plot' in 'mbcsToSbcs': dot substituted for <80>”
Warning message

In [12]:
tsne_dir <- "demux_tSNE_plots"
dir.create(tsne_dir, showWarnings = FALSE)

for (tissue in names(seurat_list)) {

  sobj <- seurat_list[[tissue]]
  cat("HTO tSNE for", tissue, "...")

  # Remove negatives (keep singlets + doublets)
  Idents(sobj) <- "HTO_classification.global"
  sobj_sub <- subset(sobj, idents = "Negative", invert = TRUE)
  n_cells <- ncol(sobj_sub)

  if (n_cells < 50) {
    cat(" skipped (only", n_cells, "cells after removing negatives)\n")
    next
  }

  # Scale + PCA on HTO features
  DefaultAssay(sobj_sub) <- "HTO"
  hto_feats <- rownames(sobj_sub[["HTO"]])
  n_htos <- length(hto_feats)

  sobj_sub <- ScaleData(sobj_sub, features = hto_feats, verbose = FALSE)

  # Max PCs = n_features - 1
  n_pcs <- min(n_htos - 1, 8)

  if (n_pcs < 2) {
    # Skin has only 2 HTOs → 1 PC → can't run tSNE with dims > 1
    # Use PCA plot instead
    sobj_sub <- RunPCA(sobj_sub, features = hto_feats, approx = FALSE, verbose = FALSE)

    pdf(file.path(tsne_dir, paste0(tissue, "_HTO_dimplot.pdf")), width = 10, height = 8)
    print(
      DimPlot(sobj_sub, reduction = "pca", group.by = "HTO_classification.global") +
        ggtitle(paste0(tissue, " - HTO PCA (2 HTOs only)"))
    )
    dev.off()
    cat(" saved PCA (only", n_htos, "HTOs)\n")

  } else {
    sobj_sub <- RunPCA(sobj_sub, features = hto_feats, approx = FALSE, verbose = FALSE)

    # Perplexity must be < n_cells/3
    perp <- min(100, floor(n_cells / 3) - 1)
    perp <- max(perp, 5)  # minimum sensible perplexity

    sobj_sub <- RunTSNE(sobj_sub, dims = 1:n_pcs, perplexity = perp,
                        check_duplicates = FALSE, verbose = FALSE)

    pdf(file.path(tsne_dir, paste0(tissue, "_HTO_tSNE.pdf")), width = 10, height = 8)

    # Color by classification (singlet vs doublet)
    print(
      DimPlot(sobj_sub, reduction = "tsne", group.by = "HTO_classification.global") +
        ggtitle(paste0(tissue, " - HTO tSNE (by classification)"))
    )

    # Color by hash.ID (which HTO was assigned)
    print(
      DimPlot(sobj_sub, reduction = "tsne", group.by = "hash.ID") +
        ggtitle(paste0(tissue, " - HTO tSNE (by hash ID)"))
    )

    dev.off()
    cat(" saved tSNE\n")
  }
}

HTO tSNE for bm ... saved tSNE
HTO tSNE for liver ... saved tSNE
HTO tSNE for lung ...

ERROR: Error in WhichCells.Seurat(object = x, cells = cells, idents = idents, : Cannot find the following identities in the object: Negative


## Save pre-removal data

In [13]:
hto_to_sample <- c(
  "TotalSeq-C0301-anti-mouse-Hashtag-1-Antibody" = "C0301",
  "TotalSeq-C0302-anti-mouse-Hashtag-2-Antibody" = "C0302",
  "TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody" = "C0303",
  "Doublet"  = "Doublet",
  "Negative" = "Negative"
)

for (tissue in names(seurat_list)) {
  sobj <- seurat_list[[tissue]]
  sobj$sample_id <- unname(hto_to_sample[sobj$hash.ID])
  sobj$condition <- paste(sobj$tissue, sobj$sample_id, sep = "_")

  # Only warn if there are truly unmapped hash.IDs (not Doublet/Negative)
  unmapped <- is.na(sobj$sample_id)
  if (any(unmapped)) {
    cat("Warning:", tissue, "has", sum(unmapped), "cells with unknown hash.IDs:\n")
    print(table(sobj$hash.ID[unmapped]))
  }

  DefaultAssay(sobj) <- "RNA"
  seurat_list[[tissue]] <- sobj
}

In [14]:
unique(seurat_list[["bm"]]$hash.ID)

[1] TotalSeq-C0301-anti-mouse-Hashtag-1-Antibody
[2] TotalSeq-C0302-anti-mouse-Hashtag-2-Antibody
[3] Negative                                    
[4] TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody
[5] Doublet                                     
5 Levels: Doublet ... TotalSeq-C0303-anti-mouse-Hashtag-3-Antibody

In [15]:
add_ids <- names(seurat_list)
merged_prefilter <- merge(seurat_list[[1]],
                          y = seurat_list[2:length(seurat_list)],
                          add.cell.ids = add_ids,
                          project = "MultiTissue_PreFilter")

cat("Pre-filter merged:", ncol(merged_prefilter), "cells x", nrow(merged_prefilter), "genes\n")
cat("\nClassification summary (all tissues):\n")
print(table(merged_prefilter$HTO_classification.global))
cat("\nPer-tissue breakdown:\n")
print(table(merged_prefilter$tissue, merged_prefilter$HTO_classification.global))

# Save RDS
dir.create("../1_outputs/0_rds", recursive = TRUE, showWarnings = FALSE)
saveRDS(merged_prefilter, file = "../1_outputs/0_rds/merged_prefilter_seurat.rds")

# Export for scanpy
prefilter_export_dir <- "scanpy_export_prefilter"
dir.create(prefilter_export_dir, showWarnings = FALSE)

DefaultAssay(merged_prefilter) <- "RNA"
if (packageVersion("Seurat") >= "5.0.0") {
  cat("Joining layers for pre-filter export...\n")
  merged_prefilter <- JoinLayers(merged_prefilter)
}

counts_pre <- GetAssayData(merged_prefilter, layer = "counts")
writeMM(counts_pre, file = file.path(prefilter_export_dir, "counts.mtx"))
cat("Saved pre-filter counts:", nrow(counts_pre), "genes x", ncol(counts_pre), "cells\n")

write.csv(data.frame(gene = rownames(counts_pre)),
          file = file.path(prefilter_export_dir, "genes.csv"),
          row.names = FALSE)

write.csv(data.frame(barcode = colnames(counts_pre)),
          file = file.path(prefilter_export_dir, "barcodes.csv"),
          row.names = FALSE)

write.csv(merged_prefilter@meta.data,
          file = file.path(prefilter_export_dir, "metadata.csv"),
          row.names = TRUE)

Pre-filter merged: 36659 cells x 20982 genes

Classification summary (all tissues):

 Doublet Negative  Singlet 
    2199     1242    33218 

Per-tissue breakdown:
        
         Doublet Negative Singlet
  bm         344       46    4273
  liver      186       28    3475
  lung        12        0     765
  skin         2        5     782
  spleen     819      692   12108
  thymus     836      471   11815
Joining layers for pre-filter export...


NULL

Saved pre-filter counts: 20982 genes x 36659 cells


## Remove Doublets

In [16]:
for (tissue in names(seurat_list)) {

  sobj <- seurat_list[[tissue]]
  n_before <- ncol(sobj)

  sobj <- subset(sobj, subset = HTO_classification.global == "Singlet")

  n_after <- ncol(sobj)
  cat(tissue, ": ", n_before, " -> ", n_after, " cells (",
      n_before - n_after, " removed)\n", sep = "")

  DefaultAssay(sobj) <- "RNA"
  sobj[["HTO"]] <- NULL

  seurat_list[[tissue]] <- sobj
}

bm: 4663 -> 4273 cells (390 removed)
liver: 3689 -> 3475 cells (214 removed)
lung: 777 -> 765 cells (12 removed)
skin: 789 -> 782 cells (7 removed)
spleen: 13619 -> 12108 cells (1511 removed)
thymus: 13122 -> 11815 cells (1307 removed)


## Save Seurat Obj

In [17]:
merged <- merge(seurat_list[[1]],
                y = seurat_list[2:length(seurat_list)],
                add.cell.ids = add_ids,
                project = "MultiTissue_Merged")

cat("Post-filter merged:", ncol(merged), "cells x", nrow(merged), "genes\n")
cat("\nSample metadata summary:\n")
print(table(merged$tissue, merged$sample_id, dnn = c("Tissue", "Sample")))

Post-filter merged: 33218 cells x 20982 genes

Sample metadata summary:
        Sample
Tissue   C0302 C0303 Doublet Negative
  bm      1641  1453       0     1179
  liver   2278  1197       0        0
  lung     497   268       0        0
  skin     241   541       0        0
  spleen  3871  4013    4224        0
  thymus  3999  3619    4197        0


In [18]:
dir.create("../1_outputs/0_rds", recursive = TRUE, showWarnings = FALSE)
saveRDS(merged, file = "../1_outputs/0_rds/merged_demuxed_seurat.rds")
saveRDS(seurat_list, file = "../1_outputs/0_rds/per_pool_seurat_list.rds")

## Convert to h5ad for scanpy

In [18]:
merged <- readRDS("../1_outputs/0_rds/merged_demuxed_seurat.rds")

In [19]:
postfilter_export_dir <- "scanpy_export"
dir.create(postfilter_export_dir, showWarnings = FALSE)

DefaultAssay(merged) <- "RNA"
if (packageVersion("Seurat") >= "5.0.0") {
  cat("Joining layers for post-filter export...\n")
  merged <- JoinLayers(merged)
}

counts <- GetAssayData(merged, layer = "counts")
writeMM(counts, file = file.path(postfilter_export_dir, "counts.mtx"))
cat("Saved post-filter counts:", nrow(counts), "genes x", ncol(counts), "cells\n")

write.csv(data.frame(gene = rownames(counts)),
          file = file.path(postfilter_export_dir, "genes.csv"),
          row.names = FALSE)

write.csv(data.frame(barcode = colnames(counts)),
          file = file.path(postfilter_export_dir, "barcodes.csv"),
          row.names = FALSE)

write.csv(merged@meta.data,
          file = file.path(postfilter_export_dir, "metadata.csv"),
          row.names = TRUE)

Joining layers for post-filter export...


NULL

Saved post-filter counts: 20982 genes x 33218 cells


In [20]:
dim(counts)

[1] 20982 33218

In [21]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /coh_labs/mvandenbrink/users/pkaur/miniconda3/envs/seurat/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: America/Phoenix
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] Matrix_1.6-5       ggplot2_3.5.2      patchwork_1.3.2    Seurat_5.3.0      
[5] SeuratObject_5.2.0 sp_2.2-0           dplyr_1.1.4       

loaded via a namespace (and not attached):
